<a href="https://colab.research.google.com/github/mannduuu07-png/urban-fire-risk-analysiskorea-fire-frequency-severity-analysis/blob/main/notebooks/03_frequency_vs_severity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - Frequency vs Severity

Core question: do regions with more fires also have higher
per-fire casualty rates? Uses (시도, 시군구) tuples throughout --
see `04_robustness_checks.ipynb` for why district names alone
are unreliable.

In [1]:
import pandas as pd
from scipy.stats import spearmanr

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

df = pd.read_parquet('/content/drive/MyDrive/fire_data/processed/cleaned_fire_data.parquet')

Mounted at /content/drive


In [2]:
def region_stats_excl_worst(data, min_count=100):
    """Per-region fire count and casualties-per-100-fires, both
    including and excluding each region's single worst incident
    (to check the result isn't driven by one mass-casualty event)."""
    result = []
    for (sido, sigungu), group in data.groupby(['시도', '시군구']):
        total_count = len(group)
        if total_count < min_count:
            continue
        total_casualty = group['인명피해(명)소계'].sum()
        worst_idx = group['인명피해(명)소계'].idxmax()
        worst_val = group.loc[worst_idx, '인명피해(명)소계']
        casualty_excl = total_casualty - worst_val
        result.append({
            '시도': sido, '시군구': sigungu, '화재건수': total_count,
            'casualties_per_100_incl': total_casualty / total_count * 100,
            'casualties_per_100_excl': casualty_excl / (total_count - 1) * 100,
            'worst_incident_casualties': worst_val,
        })
    return pd.DataFrame(result)

def top_n_tuples(region_df, col, n=10):
    top = region_df.nlargest(n, col)
    return set(zip(top['시도'], top['시군구']))

## Primary analysis: 300+ cumulative fires (stated sampling threshold)

In [3]:
region_main = region_stats_excl_worst(df, min_count=300)
national_avg = df['인명피해(명)소계'].sum() / len(df) * 100

corr, p = spearmanr(region_main['화재건수'], region_main['casualties_per_100_excl'])
print(f"Regions analyzed: {len(region_main)}")
print(f"National avg casualties per 100 fires: {national_avg:.2f}")
print(f"Spearman correlation (primary, min_count=300): rho={corr:.3f}, p={p:.4f}")

Regions analyzed: 263
National avg casualties per 100 fires: 5.76
Spearman correlation (primary, min_count=300): rho=0.100, p=0.1071


## Full-sample sensitivity check (min_count=100)

In [4]:
region_all = region_stats_excl_worst(df, min_count=100)
corr_incl, p_incl = spearmanr(region_all['화재건수'], region_all['casualties_per_100_incl'])
corr_excl, p_excl = spearmanr(region_all['화재건수'], region_all['casualties_per_100_excl'])
print(f"Including worst incident: rho={corr_incl:.3f}, p={p_incl:.4f}")
print(f"Excluding worst incident: rho={corr_excl:.3f}, p={p_excl:.4f}")

Including worst incident: rho=0.093, p=0.1156
Excluding worst incident: rho=0.155, p=0.0086


## Interpretation

Fire frequency and per-fire casualty rate show, at most, a weak
positive correlation that is not consistently significant across
sampling thresholds. This means frequency alone is not a reliable
predictor of casualty risk -- setting up the excess-risk analysis
in `05_excess_risk_analysis.ipynb`.